# Ensamblado con Protocolo Completamente Limpio

**Problema (punto 4 del revisor):** los pesos del ensamblado se optimizaban sobre validación (correcto), pero la **selección de qué 3 modelos entran** se hacía mirando el conjunto de prueba (incorrecto). Inconsistencia dentro del mismo protocolo.

**Corrección:** ambas decisiones se toman ahora sobre **validación**. El conjunto de prueba queda intacto hasta la evaluación final única.

| Paso | Antes | Ahora |
|---|---|---|
| Selección del Top-3 | Conjunto de **prueba** ❌ | Conjunto de **validación** ✓ |
| Optimización de pesos | Conjunto de validación ✓ | Conjunto de validación ✓ |
| Evaluación final | Conjunto de prueba | Conjunto de prueba |

Entrena las 9 arquitecturas, selecciona el Top-3 por **val_acc**, optimiza pesos en validación y reporta una única vez sobre test.

**Tiempo:** ~1.5-2 h. GPU T4.

In [ ]:
!pip install -q kaggle timm scikit-learn
import torch
print('GPU:', torch.cuda.is_available())

In [ ]:
from google.colab import files
import os
print('Sube tu kaggle.json:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
for fn in uploaded.keys(): os.rename(fn, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/data --unzip
print('Dataset listo')

In [ ]:
# Misma partición 70/15/15, semilla 42
import shutil, random
from pathlib import Path
random.seed(42)
SRC=Path('/content/data/chest_xray')
ALL={'NORMAL':[],'PNEUMONIA':[]}
for split in ['train','test','val']:
    sd=SRC/split
    if not sd.exists(): continue
    for img in (sd/'NORMAL').glob('*.jpeg'): ALL['NORMAL'].append(img)
    for img in (sd/'PNEUMONIA').glob('*.jpeg'): ALL['PNEUMONIA'].append(img)
DEST=Path('/content/dataset_bin')
if DEST.exists(): shutil.rmtree(DEST)
for cls,imgs in ALL.items():
    imgs=imgs.copy(); random.shuffle(imgs)
    n=len(imgs); ntr=int(n*0.70); nv=int(n*0.15)
    for sn,si in {'train':imgs[:ntr],'val':imgs[ntr:ntr+nv],'test':imgs[ntr+nv:]}.items():
        od=DEST/sn/cls; od.mkdir(parents=True,exist_ok=True)
        for img in si: shutil.copy(img,od/img.name)
for sn in ['train','val','test']:
    print(f'  {sn}:', {c: len(list((DEST/sn/c).glob("*"))) for c in ALL})

In [ ]:
import timm, numpy as np, time, json
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

device=torch.device('cuda')
NUM_CLASSES=2; BATCH=32; EPOCHS=10; LR=1e-4; PATIENCE=4

# Las 9 arquitecturas con su resolución nativa
ARCHS = {
    'InceptionV3':      ('inception_v3', 299),
    'VGG16':            ('vgg16', 224),
    'ConvNeXt-Tiny':    ('convnext_tiny', 224),
    'Swin-T':           ('swin_tiny_patch4_window7_224', 224),
    'ResNet50':         ('resnet50', 224),
    'EfficientNetV2-S': ('tf_efficientnetv2_s', 224),
    'DenseNet121':      ('densenet121', 224),
    'ViT-B/16':         ('vit_base_patch16_224', 224),
    'EfficientNet-B4':  ('efficientnet_b4', 224),
}

def loaders(img_size):
    train_tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(10),transforms.ColorJitter(brightness=0.15,contrast=0.15),
        transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    eval_tf=transforms.Compose([transforms.Resize((img_size,img_size)),transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    tr=datasets.ImageFolder('/content/dataset_bin/train',transform=train_tf)
    va=datasets.ImageFolder('/content/dataset_bin/val',transform=eval_tf)
    te=datasets.ImageFolder('/content/dataset_bin/test',transform=eval_tf)
    return (DataLoader(tr,batch_size=BATCH,shuffle=True,num_workers=2),
            DataLoader(va,batch_size=BATCH,shuffle=False,num_workers=2),
            DataLoader(te,batch_size=BATCH,shuffle=False,num_workers=2))

def get_probs(model, loader):
    model.eval(); probs,labels=[],[]
    with torch.no_grad():
        for x,y in loader:
            probs.extend(torch.softmax(model(x.to(device)),1).cpu().numpy()); labels.extend(y.numpy())
    return np.array(probs), np.array(labels)

def train_one(name, tname, isz):
    torch.manual_seed(42); np.random.seed(42)
    tr,va,te = loaders(isz)
    if tname=='inception_v3':
        m=timm.create_model(tname,pretrained=True,num_classes=2,aux_logits=False).to(device)
    else:
        m=timm.create_model(tname,pretrained=True,num_classes=2).to(device)
    crit=nn.CrossEntropyLoss(); opt=optim.AdamW(m.parameters(),lr=LR,weight_decay=0.01)
    sch=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    best,bstate,noimp=0.0,None,0
    print(f'\n--- {name} ---')
    for ep in range(EPOCHS):
        t0=time.time(); m.train()
        for x,y in tr:
            x,y=x.to(device),y.to(device)
            opt.zero_grad(); crit(m(x),y).backward(); opt.step()
        sch.step()
        pv, lv = get_probs(m, va)
        vacc = accuracy_score(lv, pv.argmax(1))
        print(f'  Ep {ep+1}/{EPOCHS} val={vacc:.4f} ({time.time()-t0:.0f}s)')
        if vacc>best: best=vacc; bstate={k:v.cpu().clone() for k,v in m.state_dict().items()}; noimp=0
        else:
            noimp+=1
            if noimp>=PATIENCE: print(f'  Early stop ep {ep+1}'); break
    if bstate: m.load_state_dict(bstate)
    pv, lv = get_probs(m, va)   # probabilidades en VALIDACIÓN
    pt, lt = get_probs(m, te)   # probabilidades en TEST
    val_acc = accuracy_score(lv, pv.argmax(1))
    test_acc = accuracy_score(lt, pt.argmax(1))
    print(f'  >> val_acc={val_acc*100:.2f}% | test_acc={test_acc*100:.2f}%')
    del m; torch.cuda.empty_cache()
    return pv, lv, pt, lt, val_acc, test_acc

## Entrenar las 9 arquitecturas y guardar probabilidades en VAL y TEST

In [ ]:
probs_val, probs_test = {}, {}
val_accs, test_accs = {}, {}
labels_val = labels_test = None

for name,(tname,isz) in ARCHS.items():
    pv, lv, pt, lt, vacc, tacc = train_one(name, tname, isz)
    probs_val[name]=pv; probs_test[name]=pt
    val_accs[name]=vacc; test_accs[name]=tacc
    if labels_val is None: labels_val=lv; labels_test=lt

print('\n' + '='*60)
print('RANKING POR VALIDACIÓN (el que decide el Top-3)')
print('='*60)
for n,v in sorted(val_accs.items(), key=lambda x:-x[1]):
    print(f'  {n:<20} val={v*100:.2f}%  (test={test_accs[n]*100:.2f}%)')

## Selección del Top-3 usando VALIDACIÓN (la corrección clave)

In [ ]:
from itertools import product
import pandas as pd

# TOP-3 SEGÚN VALIDACIÓN (no test) <-- ESTA ES LA CORRECCIÓN
top3 = [n for n,_ in sorted(val_accs.items(), key=lambda x:-x[1])[:3]]
print(f'Top-3 seleccionado por VALIDACIÓN: {top3}')
print(f'  (val_acc: ' + ', '.join(f'{n}={val_accs[n]*100:.2f}%' for n in top3) + ')')

# Para comparar: qué top-3 habría salido mirando test (protocolo incorrecto)
top3_test = [n for n,_ in sorted(test_accs.items(), key=lambda x:-x[1])[:3]]
print(f'\n(Referencia) Top-3 que habría dado el protocolo incorrecto (test): {top3_test}')
print(f'¿Coinciden? {"SÍ" if set(top3)==set(top3_test) else "NO"}')

# Optimizar pesos SOBRE VALIDACIÓN
Pv = [probs_val[n] for n in top3]
best_w, best_vacc = None, 0
for w1,w2 in product(np.arange(0,1.05,0.05), repeat=2):
    w3 = 1-w1-w2
    if w3 < -1e-9: continue
    w3 = max(w3,0)
    ens = w1*Pv[0] + w2*Pv[1] + w3*Pv[2]
    a = accuracy_score(labels_val, ens.argmax(1))
    if a > best_vacc: best_vacc, best_w = a, (round(w1,2), round(w2,2), round(w3,2))
print(f'\nPesos óptimos (sobre validación): {top3} = {best_w}')
print(f'  Exactitud en validación: {best_vacc*100:.2f}%')

# EVALUACIÓN FINAL ÚNICA SOBRE TEST
Pt = [probs_test[n] for n in top3]
w1,w2,w3 = best_w
ens_test = w1*Pt[0] + w2*Pt[1] + w3*Pt[2]
pred = ens_test.argmax(1)

acc=accuracy_score(labels_test,pred)
prec=precision_score(labels_test,pred,average='macro',zero_division=0)
rec=recall_score(labels_test,pred,average='macro',zero_division=0)
f1=f1_score(labels_test,pred,average='macro',zero_division=0)
cm=confusion_matrix(labels_test,pred); tn,fp=cm[0,0],cm[0,1]
spec=tn/(tn+fp)

print('\n' + '='*60)
print('ENSAMBLADO — PROTOCOLO COMPLETAMENTE LIMPIO')
print('='*60)
print(f'  Exactitud:     {acc*100:.2f}%')
print(f'  Precisión:     {prec*100:.2f}%')
print(f'  Sensibilidad:  {rec*100:.2f}%')
print(f'  Especificidad: {spec*100:.2f}%')
print(f'  F1:            {f1*100:.2f}%')
print(f'\n  Matriz de confusión: {cm.tolist()}')
print(classification_report(labels_test, pred, target_names=['NORMAL','PNEUMONIA'], zero_division=0))

print(f'\n¿El ensamblado supera a sus componentes?')
for n in top3:
    print(f'  {n}: {test_accs[n]*100:.2f}%')
print(f'  ENSAMBLADO: {acc*100:.2f}%  {"<- SÍ los supera" if acc > max(test_accs[n] for n in top3) else "<- NO los supera"}')

In [ ]:
# Guardar todo
salida = {
    'protocolo': 'Top-3 y pesos seleccionados sobre VALIDACION; test usado una sola vez',
    'val_acc_todas': {k: float(v) for k,v in val_accs.items()},
    'test_acc_todas': {k: float(v) for k,v in test_accs.items()},
    'top3_por_validacion': top3,
    'top3_que_habria_dado_test': top3_test,
    'coinciden': bool(set(top3)==set(top3_test)),
    'pesos_optimos': list(best_w),
    'acc_validacion_ensamblado': float(best_vacc),
    'ensamblado_test': {
        'accuracy': float(acc), 'precision': float(prec), 'recall_sensitivity': float(rec),
        'f1': float(f1), 'specificity': float(spec), 'confusion_matrix': cm.tolist()
    }
}
with open('/content/ensamblado_protocolo_limpio.json','w') as f:
    json.dump(salida, f, indent=2)

rows=[{'Arquitectura':n, 'Val (%)':round(val_accs[n]*100,2), 'Test (%)':round(test_accs[n]*100,2)}
      for n in sorted(val_accs, key=lambda x:-val_accs[x])]
rows.append({'Arquitectura':f'ENSAMBLADO ({"+".join(top3)})', 'Val (%)':round(best_vacc*100,2), 'Test (%)':round(acc*100,2)})
df=pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv('/content/ensamblado_protocolo_limpio.csv', index=False)

files.download('/content/ensamblado_protocolo_limpio.json')
files.download('/content/ensamblado_protocolo_limpio.csv')
print('\nPasame los 2 archivos.')